# Haptic VQ-VAE Colab Training

Train a temporal VQ-VAE codec on 0.5s haptic waveform segments, then evaluate reconstruction quality on the fixed validation split. This notebook does not use text labels, LLMs, PCA, or global VAE latents.

**Sections:**
1. Setup
2. Configure VQ-VAE run
3. Train
4. Evaluate reconstruction
5. Inspect codebook usage


In [ ]:
# 1. Clone code repo (or update existing checkout)
import os
import subprocess
import sys

REPO_URL = "https://github.com/cindy-77jiayi/thesis_hapticAE.git"
REPO_DIR = "/content/thesis_hapticAE"
BRANCH = "vae-withoutpca"
FORCE_CLEAN_CLONE = False

if FORCE_CLEAN_CLONE and os.path.exists(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=False)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"Repo ready at: {REPO_DIR}")


In [ ]:
# 2. Clone dataset
DATASET_URL = "https://github.com/HapticGen/hapticgen-dataset.git"
DATASET_DIR = "/content/hapticgen-dataset"
FORCE_DATASET_REFRESH = False

if FORCE_DATASET_REFRESH and os.path.exists(DATASET_DIR):
    subprocess.run(["rm", "-rf", DATASET_DIR], check=True)

if not os.path.exists(os.path.join(DATASET_DIR, ".git")):
    subprocess.run(["git", "clone", DATASET_URL, DATASET_DIR], check=True)
else:
    subprocess.run(["git", "-C", DATASET_DIR, "pull", "--ff-only"], check=False)

print(f"Dataset ready at: {DATASET_DIR}")


In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt


In [ ]:
# 4. Configure VQ-VAE run
from src.utils.config import load_config
import yaml

CONFIG = "configs/vqvae_0p5s.yaml"
DATA_DIR = DATASET_DIR
OUTPUT_DIR = "/content/outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
config = load_config(CONFIG)

print(f"CONFIG: {CONFIG}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print("\nKey fields:")
print(f"  model_type: {config['model_type']}")
print(f"  run_name: {config['run_name']}")
print(f"  T: {config['data']['T']}")
print(f"  batch_size: {config['training']['batch_size']}")
print(f"  channels: {config['model']['channels']}")
print(f"  embedding_dim: {config['vq']['embedding_dim']}")
print(f"  codebook_size: {config['vq']['codebook_size']}")
print(f"  commitment_cost: {config['vq']['commitment_cost']}")

print("\nFull resolved config:")
print(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))


In [ ]:
# 5. Run training
!python scripts/train.py --config $CONFIG --data_dir $DATA_DIR --output_dir $OUTPUT_DIR


In [ ]:
# 6. Load checkpoint and fixed validation split
import json
import numpy as np
import torch

from src.data.loaders import build_dataloaders, build_model, load_checkpoint

config = load_config(CONFIG)
run_dir = os.path.join(OUTPUT_DIR, config["run_name"])
best_path = os.path.join(run_dir, "best_model.pt")
split_manifest_path = os.path.join(run_dir, "data_split.json")

assert os.path.exists(best_path), f"Missing best checkpoint: {best_path}"
assert os.path.exists(split_manifest_path), f"Missing split manifest: {split_manifest_path}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = build_dataloaders(config, DATA_DIR, split_manifest_path=split_manifest_path)
model = build_model(config, device=device)
model = load_checkpoint(model, best_path, device=device)

print(f"Loaded: {best_path}")
print(f"Device: {device}")
print(f"Train batches: {len(data['train_loader'])}, val batches: {len(data['val_loader'])}")
print(f"Fixed split: {split_manifest_path}")


In [ ]:
# 7. Plot training curve
import matplotlib.pyplot as plt

history_path = os.path.join(run_dir, "history.json")
with open(history_path, "r", encoding="utf-8") as f:
    history = json.load(f)

epochs = [item["epoch"] for item in history]
train_loss = [item["train"]["loss"] for item in history]
val_loss = [item["val"]["loss"] for item in history]
vq_loss = [item["train"].get("vq", 0.0) for item in history]
perplexity = [item["train"].get("vq_perplexity", 0.0) for item in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(epochs, train_loss, label="train")
axes[0].plot(epochs, val_loss, label="val")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs, vq_loss, label="vq loss")
axes[1].plot(epochs, perplexity, label="perplexity")
axes[1].set_title("VQ Codebook Metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Best val: {min(val_loss):.6f} at epoch {epochs[int(np.argmin(val_loss))]}")


In [ ]:
# 8. Evaluate reconstruction on fixed validation set
from src.eval.evaluate import evaluate_reconstruction, print_metrics
from src.eval.visualize import plot_waveform_comparison

result = evaluate_reconstruction(
    model,
    data["val_loader"],
    device,
    n_samples=8,
    is_vae=False,
    is_vqvae=True,
    sr=config["data"]["sr"],
    clamp_range=config["loss"].get("clamp_range", None),
)
print_metrics(result)

ratios = [m["std_ratio"] for m in result["per_sample"]]
print(f"\nVQ-VAE mean STD ratio: {np.mean(ratios):.1%}")
plot_waveform_comparison(
    result["x_np"],
    result["xhat_np"],
    n_show=min(8, len(result["x_np"])),
    title_prefix="VQ-VAE fixed-val recon",
)


In [ ]:
# 9. Inspect codebook usage on the evaluation batch
codes = result["codes"]
flat_codes = codes.reshape(-1)
unique_codes, counts = np.unique(flat_codes, return_counts=True)

print(f"Code token shape: {codes.shape}")
print(f"Unique codes used in eval batch: {len(unique_codes)} / {config['vq']['codebook_size']}")
print(f"Eval perplexity: {result['vq_perplexity']:.2f}")

top = sorted(zip(unique_codes.tolist(), counts.tolist()), key=lambda x: x[1], reverse=True)[:20]
print("Top code ids:")
for code_id, count in top:
    print(f"  code {code_id:4d}: {count}")

plt.figure(figsize=(12, 3))
plt.bar(unique_codes, counts, width=1.0)
plt.title("Codebook usage on evaluation batch")
plt.xlabel("Code id")
plt.ylabel("Count")
plt.tight_layout()
plt.show()
